In [1]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

print("임베딩 모델을 로드하는 중입니다...")

# 1. 임베딩 모델 준비 (DB 저장 시 사용했던 모델과 완전히 동일해야 함)
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# 2. 하드디스크에 저장된 Chroma DB 폴더 연결
vectorstore = Chroma(
    persist_directory="./insurance_chroma_db", 
    embedding_function=embeddings
)

print("✅ 저장된 Vector DB를 성공적으로 불러왔습니다!")

# (보너스) DB가 제대로 불러와졌는지 저장된 조각(Chunk)의 총 개수 확인
try:
    count = vectorstore._collection.count()
    print(f"📊 현재 DB에 저장된 약관 조각 총 개수: {count}개")
except Exception as e:
    print("DB 연결 확인 완료 (조각 개수는 숨겨져 있음)")

임베딩 모델을 로드하는 중입니다...


c:\Users\cloud\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED



✅ 저장된 Vector DB를 성공적으로 불러왔습니다!
📊 현재 DB에 저장된 약관 조각 총 개수: 1344개


In [2]:
## 약관 검색해보기 

# 고객의 질문 (자연어)
query = "음주운전 중에 사고가 났는데, 내가 부담해야 하는 면책금이 얼마야?"

print(f"\n고객 질문: '{query}'")
print("가장 유사한 약관 조각 2개를 찾아옵니다...\n")

# DB에서 질문과 가장 의미가 비슷한 조각 2개(k=2) 검색
retrieved_docs = vectorstore.similarity_search(query, k=2)

for i, doc in enumerate(retrieved_docs):
    print(f"--- [검색된 조각 {i+1}] ---")
    print(f"📍 출처(메타데이터): {doc.metadata}")
    print(f"📄 내용:\n{doc.page_content}\n")


고객 질문: '음주운전 중에 사고가 났는데, 내가 부담해야 하는 면책금이 얼마야?'
가장 유사한 약관 조각 2개를 찾아옵니다...

--- [검색된 조각 1] ---
📍 출처(메타데이터): {'대분류': 'Q. 무면허운전이나 음주운전 시 발생한 사고도 보상받을 수 있나요?'}
📄 내용:
# Q. 무면허운전이나 음주운전 시 발생한 사고도 보상받을 수 있나요?  
A. 무면허운전이나 음주운전 사고 시 제한적으로 보상이 가능하며, 본인이 부담해야하는 금액이 높으므로 주의하시기 바랍니다.  
| 구분      | 대인 배상Ⅰ | 대인 배상Ⅱ | 대물배상 (의무) | 대물배상 (임의) | 자기신체 사고 | 무보험차 | 자기차량 손해 |
| ------- | ------ | ------ | --------- | --------- | ------- | ---- | ------- |
|         | 음주     | 사고     | 부담금       | 사고        | 부담금     | 보상   | 보상      |
| 마약 약물운전 | 사고     | 부담금    | 지급        | 부담금       | 보상      | 보상   |         |
| 무면허     | 전액     |        | 전액        |           | 면책      |      |         |
| 뺑소니     |        |        | 보상        |           |         |      |         |

--- [검색된 조각 2] ---
📍 출처(메타데이터): {'대분류': '제11조 (음주운전, 무면허운전, 마약·약물운전 또는 사고발생 시의 조치 의무 위반 관련 사고부담금)'}
📄 내용:
# 제11조 (음주운전, 무면허운전, 마약·약물운전 또는 사고발생 시의 조치 의무 위반 관련 사고부담금)  
................ 60



# LLM 결합

In [3]:
# pip install -q -U google-genai

In [4]:
# pip install langchain-google-genai

In [5]:
# pip install --upgrade langchain-google-genai google-generativeai

In [ ]:
## LLM 동작 테스트

from langchain_google_genai import ChatGoogleGenerativeAI

# 1. 뇌(LLM) 세팅 (최신 버전 패키지에서는 그냥 'gemini-1.5-flash'로 씁니다)
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)  # GOOGLE_API_KEY 환경변수에서 자동으로 읽음

# 2. 아주 단순한 질문 던져보기 (생존 신고)
print("Gemini에게 인사하는 중...")
response = llm.invoke("안녕! 너 지금 잘 작동하고 있어? 10자 이내로 대답해봐.")

print("💡 [Gemini의 답변]:", response.content)

In [12]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 3. 프롬프트 세팅
prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 정확하고 보수적인 자동차보험 보상 안내 어시스턴트입니다.
반드시 제공된 <context> 안의 내용만을 기반으로 답변하십시오.
표의 형태가 조금 쪼개져 있더라도 문맥을 논리적으로 추론하여 정확한 금액이나 조건을 안내해야 합니다.
<context>에 없는 내용이라면 절대 지어내지 마십시오.

<context>
{context}
</context>"""),
    ("user", "{question}")
])

chain = prompt | llm | StrOutputParser()

# 테스트

In [ ]:
# ==========================================
# 🎯 실제 고객 응대 테스트
# ==========================================
query = "음주운전 중에 사고가 났는데, 내가 부담해야 하는 면책금이 얼마야?"
print(f"\n고객 질문: '{query}'")
print("2. 약관 DB 검색 및 답변을 생성 중입니다...\n")

# A. DB에서 약관 조각 찾아오기
retrieved_docs = vectorstore.similarity_search(query, k=4)
context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

# B. Gemini가 답변 작성
response = chain.invoke({
    "context": context_text,
    "question": query
})

print("====================================")
print("💡 [약관 에이전트(Gemini) 최종 답변]")
print(response)
print("====================================")

In [11]:
# ==========================================
# 🎯 실제 고객 응대 테스트
# ==========================================
query = "태풍으로 차가 침수됐는데, 자기차량손해 담보로 보상받을 수 있어?"
print(f"\n고객 질문: '{query}'")
print("2. 약관 DB 검색 및 답변을 생성 중입니다...\n")

# A. DB에서 약관 조각 찾아오기
retrieved_docs = vectorstore.similarity_search(query, k=4)
context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

# B. Gemini가 답변 작성
response = chain.invoke({
    "context": context_text,
    "question": query
})

print("====================================")
print("💡 [약관 에이전트(Gemini) 최종 답변]")
print(response)
print("====================================")


고객 질문: '태풍으로 차가 침수됐는데, 자기차량손해 담보로 보상받을 수 있어?'
2. 약관 DB 검색 및 답변을 생성 중입니다...

💡 [약관 에이전트(Gemini) 최종 답변]
네, 태풍으로 인한 차량 침수 피해는 자기차량손해 담보의 '[53] 자기차량손해 침수·화재 피해 한정 보상 특별약관'에 따라 보상받으실 수 있습니다.

다만, 침수로 인정되는 조건이 있습니다.
*   **침수**는 흐르거나 고인 물, 역류하는 물, 범람하는 물, 해수 등에 피보험자동차가 빠지거나 잠기는 것을 의미합니다.
*   **제외되는 경우:** 차량 도어나 선루프 등을 개방해 놓았을 때 빗물이 들어간 것은 침수로 보지 않습니다.

침수로 인해 '차량단독사고 손해보상 특별약관'에 의한 보험금이 지급되는 경우 다음과 같은 보상도 추가로 받으실 수 있습니다.

1.  **차량 하체보호(언더코팅) 비용 보상 (전부손해 제외)**
    *   피보험자동차의 하체보호(언더코팅) 작업을 시공한 경우, 실제 소요된 비용을 아래 한도액 내에서 지급합니다.
        *   **소형승용차 (1,600cc 이하, 전기경형):** 200,000원
        *   **중형승용차 (1,600cc 초과~2,000cc 이하):** 250,000원
        *   **대형승용차 (2,000cc 초과):** 300,000원
        *   **다목적승용차 (법정승차정원 7~10인):** 300,000원

2.  **전부손해 시 신규 등록차량 취·등록세 및 검수비용 지급**
    *   한 번의 사고로 생긴 손해가 전부손해일 경우, 보험증권에 기재된 자기차량손해 보험가입금액의 7% 한도 내에서 피보험자 명의로 차량을 신규 등록할 때 실제로 소요된 취·등록세를 지급합니다.
    *   취·등록세가 지급되는 경우, 신규 등록차량 검수비용 20만원을 추가로 지급합니다.

3.  **렌트를 하지 않는 경우 교통비 지급**
    *   피보험자동차와 동급의 국내산 자동차를 대여하는 데 소요

In [13]:
# ==========================================
# 🎯 실제 고객 응대 테스트
# ==========================================
query = "저희 자동차보험이 '가족 운전자 한정 운전 특약'으로 가입되어 있는데요. 이번 명절에 사위가 제 차를 교대로 운전하다가 앞차를 살짝 박았습니다. 사위도 가족한정 범위에 포함되어서 정상적으로 보험 처리가 가능한가요?"
print(f"\n고객 질문: '{query}'")
print("2. 약관 DB 검색 및 답변을 생성 중입니다...\n")

# A. DB에서 약관 조각 찾아오기
retrieved_docs = vectorstore.similarity_search(query, k=4)
context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

# B. Gemini가 답변 작성
response = chain.invoke({
    "context": context_text,
    "question": query
})

print("====================================")
print("💡 [약관 에이전트(Gemini) 최종 답변]")
print(response)
print("====================================")


고객 질문: '저희 자동차보험이 '가족 운전자 한정 운전 특약'으로 가입되어 있는데요. 이번 명절에 사위가 제 차를 교대로 운전하다가 앞차를 살짝 박았습니다. 사위도 가족한정 범위에 포함되어서 정상적으로 보험 처리가 가능한가요?'
2. 약관 DB 검색 및 답변을 생성 중입니다...



ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 1.455523664s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi

In [ ]:
# ==========================================
# 🎯 실제 고객 응대 테스트
# ==========================================
query = "출고된 지 딱 1년 2개월 된 새 차인데, 뒤차가 신호 대기 중인 제 차를 세게 박아서 수리비가 차량 가액의 30%나 나왔습니다. 수리비 말고 차량 시세가 떨어진 것에 대한 보상(시세하락손해)도 받을 수 있나요? 받을 수 있다면 기준이 어떻게 되나요?"
print(f"\n고객 질문: '{query}'")
print("2. 약관 DB 검색 및 답변을 생성 중입니다...\n")

# A. DB에서 약관 조각 찾아오기
retrieved_docs = vectorstore.similarity_search(query, k=4)
context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

# B. Gemini가 답변 작성
response = chain.invoke({
    "context": context_text,
    "question": query
})

print("====================================")
print("💡 [약관 에이전트(Gemini) 최종 답변]")
print(response)
print("====================================")

In [ ]:
# ==========================================
# 🎯 실제 고객 응대 테스트
# ==========================================
query = "음주운전 중에 사고가 났는데, 내가 부담해야 하는 면책금이 얼마야?"
print(f"\n고객 질문: '{query}'")
print("2. 약관 DB 검색 및 답변을 생성 중입니다...\n")

# A. DB에서 약관 조각 찾아오기
retrieved_docs = vectorstore.similarity_search(query, k=4)
context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

# B. Gemini가 답변 작성
response = chain.invoke({
    "context": context_text,
    "question": query
})

print("====================================")
print("💡 [약관 에이전트(Gemini) 최종 답변]")
print(response)
print("====================================")

In [ ]:
# ==========================================
# 🎯 실제 고객 응대 테스트
# ==========================================
query = "음주운전 중에 사고가 났는데, 내가 부담해야 하는 면책금이 얼마야?"
print(f"\n고객 질문: '{query}'")
print("2. 약관 DB 검색 및 답변을 생성 중입니다...\n")

# A. DB에서 약관 조각 찾아오기
retrieved_docs = vectorstore.similarity_search(query, k=4)
context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

# B. Gemini가 답변 작성
response = chain.invoke({
    "context": context_text,
    "question": query
})

print("====================================")
print("💡 [약관 에이전트(Gemini) 최종 답변]")
print(response)
print("====================================")